In [177]:
import pickle
import pandas as pd
import numpy as np
from sklearn.covariance import LedoitWolf

## 1. Loading the Data

In [178]:
with open("hw3_input.pickle", "rb") as f:
    data = pickle.load(f)

C:\Users\Yechao Chen\AppData\Local\Temp\ipykernel_12404\4019908859.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data = pickle.load(f)


In [179]:
price = data["price"] # Txn
tri = data["tri"] # Txn
volume = data["volume"] # Txn
mtbv = data["mtbv"] # Txn
cap = data["cap"] # Txn
tcost = data["tcost"] # Txn
rec = data["rec"] # Txn
isactivenow = data["isactive"] # Txn

allstocks = price.columns.to_numpy() # 1xn
myday = price.index.to_numpy() # Tx1

# T and n
T, n = price.shape
print(f"T = {T}, n = {n}")

T = 1521, n = 411


## 2. Risk Model

In [180]:
tri_clean = tri.copy()
tri_clean = tri_clean.apply(pd.to_numeric, errors="coerce")

# Compute arithmetic daily returns manually: R_t = TRI_t / TRI_{t-1} - 1 since pct_change is causing issues
previous_tri = tri_clean.shift(1)
returns = (tri_clean / previous_tri) - 1
returns = returns.replace([np.inf, -np.inf], np.nan)
returns = returns.clip(lower=-0.99, upper=0.99)
returns = returns.fillna(0)

all_dates = returns.index

In [181]:
# Collect (year, month) pairs, set up vectors and lists for later
year_month_list = []
for dt in all_dates:
    ym = (dt.year, dt.month)
    if ym not in year_month_list:
        year_month_list.append(ym)

# shrinkage intensities (alpha) for each month
shrink_list = [] # months x 1 vector

# first trading date per month
month_ref_dates = [] # one date per month, same order as shrink_list

# covariance matrices per month
cov_by_month = {} # key: ref_date, value: covariance DataFrame

In [182]:
def get_month_reference_date(year, month, date_index):

    first_calendar_day = pd.Timestamp(year=year, month=month, day=1)
    
    # find all dates <= first_calendar_day
    mask = date_index <= first_calendar_day
    if not mask.any():
        return None
    
    # reference date is the max of those
    ref_date = date_index[mask].max()
    return ref_date

In [183]:
for (year, month) in year_month_list:
    # get reference date for this month
    ref_date = get_month_reference_date(year, month, all_dates)
    if ref_date is None:
        continue
    
    # find the position of ref_date in the index
    try:
        ref_pos = all_dates.get_loc(ref_date)
    except KeyError:
        continue
    
    # check if we have at least 252 days before this date
    if ref_pos < 252:
        continue
    
    # define the 252-day lookback window
    start_pos = ref_pos - 252
    end_pos = ref_pos  # slice is [start_pos, end_pos) in iloc
    
    returns_window = returns.iloc[start_pos:end_pos]  # shape: (252, n)
    
    # determine active stocks at the reference date
    active_row = isactivenow.loc[ref_date]
    active_mask = (active_row == 1)
    
    # list of active stocks
    active_stocks = list(isactivenow.columns[active_mask])
    
    if len(active_stocks) == 0:
        continue
    
    # restrict returns to active universe
    R = returns_window[active_stocks] # DataFrame (252 x n_active)
    
    # fill NaNs
    R_filled = R.fillna(0)
    
    # Ledoit-Wolf shrinkage
    X = R_filled.values # shape: (252, n_active)
    lw = LedoitWolf(assume_centered=False)
    
    # fit covariance matrix
    lw.fit(X)
    cov_matrix = lw.covariance_ # numpy array (n_active x n_active)
    alpha = float(lw.shrinkage_) # shrinkage intensity
    
    # if intensity is negative, set to 0
    if alpha < 0:
        alpha = 0.0
    
    # store shrinkage intensity
    shrink_list.append(alpha)
    month_ref_dates.append(ref_date)
    
    # turn covariance into a DataFrame for convenience later
    cov_df = pd.DataFrame(
        cov_matrix,
        index=active_stocks,
        columns=active_stocks
    )
    cov_by_month[ref_date] = cov_df


In [184]:
shrink = np.array(shrink_list).reshape(-1, 1)

print("Number of months with risk model:", shrink.shape[0])

#shrink

Number of months with risk model: 59


## 3. Alphas

In [185]:
def process_alpha(alpha_raw, isactivenow, winsor_limit=3.0):
    """
    For active stocks, demean, standardize, winsorize.
    For inactive stocks, we set the alpha to 0.
    """
    alpha = alpha_raw.copy().astype(float)
    
    for date in alpha.index:
        # which stocks are active on this date?
        active_mask = (isactivenow.loc[date] == 1)
        if active_mask.sum() == 0:
            # no active stocks → set whole row to 0
            alpha.loc[date, :] = 0.0
            continue
        
        # Values for active stocks
        vals = alpha.loc[date, active_mask]
        
        mean_val = vals.mean()
        std_val = vals.std()
        
        # If std is 0 or NaN, avoid division; set row to 0
        if (std_val is None) or (std_val == 0) or np.isnan(std_val):
            alpha.loc[date, :] = 0.0
            continue
        
        # z-score: (x - mean) / std
        z = (vals - mean_val) / std_val
        
        # winsorize: cap at +/- winsor_limit
        z = z.clip(lower=-winsor_limit, upper=winsor_limit)
        
        # Put back into alpha
        alpha.loc[date, active_mask] = z
        
        # For inactive stocks, set to 0
        alpha.loc[date, ~active_mask] = 0.0
    
    # Replace any remaining NaNs with 0
    alpha = alpha.fillna(0.0)
    return alpha

### SHORT-TERM CONTRARIAN

In [186]:
# Parameters
K_REV = 10  # lookback window for reversal (e.g., 10 days)

# Initialize alpharev with zeros
alpharev_raw = pd.DataFrame(0.0, index=all_dates, columns=allstocks)

# Triangular weights: [10, 9, ..., 1]
weights = list(range(1, K_REV + 1))  # [1,2,...,10]
weights = weights[::-1]              # [10,9,...,1]

for t in range(K_REV, T):
    # Take the last K_REV days of returns: rows t-K_REV ... t-1
    window = returns.iloc[t-K_REV:t]   # shape (K_REV x n)
    
    # Weighted sum across time
    weighted_sum = pd.Series(0.0, index=allstocks)
    
    for k in range(K_REV):
        day_returns = window.iloc[k]      # this is a Series for one day
        weight = weights[k]
        weighted_sum = weighted_sum + weight * day_returns
    
    # Contrarian: negative sign
    alpharev_raw.iloc[t] = -weighted_sum

# For first K_REV days, we leave alpharev_raw as 0 (no signal yet)

alpharev = process_alpha(alpharev_raw, isactivenow)
alpharev

,1COVG.DE,AALB.AS,ABB.ST,ABI.BR,ABIO.PA^J22,ABNd.AS,ACCP.PA,ACKB.BR,AD.AS,ADEA.OL^F24,...,WDIG.DE^A21,WDPP.BR,WEHA.AS,WLN.PA,WLSNc.AS,WRT1V.HE,XFAB.PA,YAR.OL,ZALG.DE,ZELA.CO
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-02,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-03,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-04,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-05,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-06,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,-0.353707,0.553697,0.084788,0.410519,0.0,-0.791654,-0.786941,1.287432,0.914685,0.0,...,0.0,1.097305,-0.699153,0.574039,0.257614,-0.408673,0.598259,0.966731,-0.804740,0.426779
2025-07-29,0.894358,0.558582,-0.392944,-0.204818,0.0,-0.384515,-1.301137,1.404022,0.474824,0.0,...,0.0,0.677529,-0.936819,0.901870,-0.224026,-0.534698,0.215124,0.576034,0.441003,0.079676
2025-07-30,0.786044,0.238956,0.839825,-0.689737,0.0,-0.835212,-1.854262,1.221259,0.205385,0.0,...,0.0,0.442044,-1.406408,0.498176,-0.674075,-0.846743,-0.020305,0.360334,0.296795,-0.202464


### SHORT-TERM PROCYCLICAL

In [187]:
H_REC = 20  # lookback window for recommendation revisions

# Raw recommendation revision: rec_t - rec_{t-20}
alpharec_raw = rec - rec.shift(H_REC)

# Replace NaNs (early days) with 0
alpharec_raw = alpharec_raw.fillna(0)

# Process it
alpharec = process_alpha(alpharec_raw, isactivenow)
alpharec

Instrument,1COVG.DE,AALB.AS,ABB.ST,ABI.BR,ABIO.PA^J22,ABNd.AS,ACCP.PA,ACKB.BR,AD.AS,ADEA.OL^F24,...,WDIG.DE^A21,WDPP.BR,WEHA.AS,WLN.PA,WLSNc.AS,WRT1V.HE,XFAB.PA,YAR.OL,ZALG.DE,ZELA.CO
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-02,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-03,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-04,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-05,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-06,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,-0.101798,-0.101798,-0.101798,-0.101798,0.0,1.877615,-1.091505,-0.101798,-0.101798,0.0,...,0.0,-1.091505,-0.101798,-1.091505,0.887908,0.887908,-0.101798,-0.101798,0.887908,-2.081212
2025-07-29,-0.107566,-0.107566,-0.107566,-0.107566,0.0,1.927464,-1.125081,-0.107566,-0.107566,0.0,...,0.0,-1.125081,-0.107566,-0.107566,0.909949,0.909949,-0.107566,-0.107566,0.909949,-2.142595
2025-07-30,-0.133384,-0.133384,-0.133384,-0.133384,0.0,1.935547,-1.167849,-0.133384,-0.133384,0.0,...,0.0,-1.167849,-0.133384,-0.133384,0.901081,0.901081,-0.133384,-0.133384,0.901081,-2.202314


### LONG-TERM CONTRARIAN

In [188]:
alphaval_raw = -mtbv.copy().astype(float)

# Handle NaNs
alphaval_raw = alphaval_raw.fillna(0)

# Process it
alphaval = process_alpha(alphaval_raw, isactivenow)
alphaval

Instrument,1COVG.DE,AALB.AS,ABB.ST,ABI.BR,ABIO.PA^J22,ABNd.AS,ACCP.PA,ACKB.BR,AD.AS,ADEA.OL^F24,...,WDIG.DE^A21,WDPP.BR,WEHA.AS,WLN.PA,WLSNc.AS,WRT1V.HE,XFAB.PA,YAR.OL,ZALG.DE,ZELA.CO
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-02,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-03,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-04,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-05,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-06,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,0.203974,0.167127,0.621530,0.075793,0.0,-1.795477,0.453150,0.023969,0.283677,0.0,...,0.0,-0.210940,-0.716464,-2.217396,0.711014,0.530437,-0.552846,-0.149512,0.476526,0.599410
2025-07-29,0.203974,0.167127,0.621530,0.075793,0.0,-1.795477,0.453150,0.023969,0.283677,0.0,...,0.0,-0.210940,-0.716464,-2.217396,0.711014,0.530437,-0.552846,-0.149512,0.476526,0.599410
2025-07-30,0.205700,0.168883,0.622919,0.077622,0.0,-1.792137,0.454675,0.025840,0.285338,0.0,...,0.0,-0.208879,-0.713995,-2.213715,0.712331,0.531899,-0.550509,-0.147500,0.478032,0.600817


### LONG-TERM PROCYCLICAL

In [189]:
SKIP_1M = 21   # days to skip (most recent month)
LOOKBACK_12M = 252  # days for 12-month window

# TRI at t-21 and t-252
tri_1m_ago = tri_clean.shift(SKIP_1M)
tri_12m_ago = tri_clean.shift(LOOKBACK_12M)

alphamom_raw = (tri_1m_ago / tri_12m_ago) - 1

# Clean NaNs and infinite values
alphamom_raw = alphamom_raw.replace([np.inf, -np.inf], np.nan)
alphamom_raw = alphamom_raw.fillna(0.0)

# Process it
alphamom = process_alpha(alphamom_raw, isactivenow)
alphamom

Instrument,1COVG.DE,AALB.AS,ABB.ST,ABI.BR,ABIO.PA^J22,ABNd.AS,ACCP.PA,ACKB.BR,AD.AS,ADEA.OL^F24,...,WDIG.DE^A21,WDPP.BR,WEHA.AS,WLN.PA,WLSNc.AS,WRT1V.HE,XFAB.PA,YAR.OL,ZALG.DE,ZELA.CO
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-02,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-03,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-04,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-05,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-06,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,0.051704,-0.098601,-0.078957,0.122306,0.0,0.109639,0.274946,0.083713,-0.159285,0.0,...,0.0,0.651695,0.646750,0.279946,-0.093449,-0.063485,-0.046062,0.040743,0.164973,-0.034679
2025-07-29,-0.173063,0.213880,0.005538,-0.250725,0.0,0.011196,-0.109731,0.097634,0.286369,0.0,...,0.0,0.259841,-0.816989,-1.449810,0.966092,0.001659,-0.058754,-0.401767,-0.319902,0.227780
2025-07-30,0.042158,0.042099,0.160982,-0.006243,0.0,0.055492,-0.010200,0.063182,-0.001575,0.0,...,0.0,0.031610,-0.052049,-0.034930,0.239062,0.121451,0.118930,0.006784,-0.061160,0.002558


### BLEND

In [190]:
alphablend_raw = (
    0.50 * alpharev +
    0.25 * alpharec +
    0.15 * alphaval +
    0.10 * alphamom
)

alphablend = process_alpha(alphablend_raw, isactivenow)
alphablend

,1COVG.DE,AALB.AS,ABB.ST,ABI.BR,ABIO.PA^J22,ABNd.AS,ACCP.PA,ACKB.BR,AD.AS,ADEA.OL^F24,...,WDIG.DE^A21,WDPP.BR,WEHA.AS,WLN.PA,WLSNc.AS,WRT1V.HE,XFAB.PA,YAR.OL,ZALG.DE,ZELA.CO
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-02,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-03,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-04,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-05,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2019-09-06,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,-0.307679,0.461839,0.169893,0.349561,0.0,-0.340091,-1.026029,1.107850,0.802780,0.0,...,0.0,0.537693,-0.754107,-0.527860,0.784259,0.149604,0.318893,0.769109,-0.175999,-0.403496
2025-07-29,0.747496,0.510628,-0.242656,-0.266258,0.0,0.022811,-1.552982,1.195668,0.480479,0.0,...,0.0,0.076316,-1.218271,-0.108972,0.545580,0.055274,-0.029123,0.334218,0.842264,-0.688435
2025-07-30,0.682124,0.189484,0.861044,-0.662914,0.0,-0.362456,-2.048154,1.022346,0.182980,0.0,...,0.0,-0.189672,-1.513173,-0.226207,0.018807,-0.202134,-0.216275,0.206607,0.760695,-1.005795


## 4 Optimizer

In [191]:
# 交易样本：2021-01-01 ~ 2025-08-01
start_date = pd.Timestamp("2021-01-01")
end_date   = pd.Timestamp("2025-08-01")

dates = returns.index

# 找到第一个 >= start_date 的交易日
start_idx = int(np.where(dates >= start_date)[0][0])
# 找到最后一个 <= end_date 的交易日
end_idx   = int(np.where(dates <= end_date)[0][-1])

# t0：第一天开始下单的 index（作业要求的标量）
t0 = start_idx
print("t0 index =", t0, "date =", dates[t0].date(), "end date =", dates[end_idx].date())


t0 index = 343 date = 2021-01-04 end date = 2025-08-01


In [192]:
# 把 month_ref_dates 转成排好序的 DatetimeIndex
month_ref_dates = pd.to_datetime(month_ref_dates)
month_ref_dates_sorted = month_ref_dates.sort_values()

def get_cov_for_date(current_date: pd.Timestamp):
    """
    给定当前日期，取最近一个 <= 当前日期 的 ref_date，
    用它对应的 shrinkage 协方差矩阵。
    返回：
      Sigma_df: DataFrame（active_stock × active_stock）
      active_stocks: 对应的股票代码 np.array
    如果没有可用的 month，则返回 (None, None)
    """
    mask = month_ref_dates_sorted <= current_date
    if not mask.any():
        return None, None
    ref_date = month_ref_dates_sorted[mask].max()
    Sigma_df = cov_by_month[ref_date]
    active_stocks = Sigma_df.index.to_numpy()
    return Sigma_df, active_stocks


In [199]:
returns_np = price.pct_change().fillna(0.0).clip(-1, 1).to_numpy(dtype=float)
alphablend_np = alphablend.fillna(0.0).to_numpy(dtype=float)
tcost_np = tcost.fillna(0.0).to_numpy(dtype=float)

active_numeric = (
    isactivenow
    .apply(pd.to_numeric, errors="coerce")
    .astype(float)           # force off BooleanArray dtype
    .fillna(0.0)
)
active_np = (active_numeric.to_numpy(dtype=float) > 0.5)

T, n = returns_np.shape


C:\Users\Yechao Chen\AppData\Local\Temp\ipykernel_12404\3143412343.py:1: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns_np = price.pct_change().fillna(0.0).clip(-1, 1).to_numpy(dtype=float)
C:\Users\Yechao Chen\AppData\Local\Temp\ipykernel_12404\3143412343.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tcost_np = tcost.fillna(0.0).to_numpy(dtype=float)


In [200]:
def build_target(alpha_vec: np.ndarray,
                 Sigma_df: pd.DataFrame,
                 active_mask: np.ndarray,
                 mu_param: float) -> np.ndarray:
    """
    根据当日 alpha、当月协方差 Σ 和活跃股票，构造 long/short 目标持仓（欧元）。
    """
    target = np.zeros_like(alpha_vec)

    stocks_in_cov = Sigma_df.index.to_numpy()
    idx_cov = pd.Index(allstocks).get_indexer(stocks_in_cov)
    mask_active_cov = active_mask[idx_cov]

    if mask_active_cov.sum() < 2:
        return target

    idx = idx_cov[mask_active_cov]
    alpha_sub = alpha_vec[idx]
    Sigma_sub = Sigma_df.loc[stocks_in_cov[mask_active_cov],
                             stocks_in_cov[mask_active_cov]].to_numpy(dtype=float)

    Sigma_sub = Sigma_sub + 1e-6 * np.eye(Sigma_sub.shape[0])

    try:
        w_dir = np.linalg.solve(Sigma_sub, alpha_sub)
    except np.linalg.LinAlgError:
        w_dir = np.linalg.pinv(Sigma_sub) @ alpha_sub

    # 强制形成 long/short
    if (w_dir > 0).all() or (w_dir < 0).all():
        order = np.argsort(alpha_sub)
        k = len(order) // 2
        long_idx = order[k:]
        short_idx = order[:k]
        w_dir = np.zeros_like(alpha_sub)
        w_dir[long_idx] = alpha_sub[long_idx]
        w_dir[short_idx] = -alpha_sub[short_idx]

    
    pos_sum = w_dir[w_dir > 0].sum()
    w_dir[w_dir > 0] = w_dir[w_dir > 0] / pos_sum
    neg_sum = -w_dir[w_dir < 0].sum()
    w_dir[w_dir < 0] = w_dir[w_dir < 0] / neg_sum
    w_dir = np.clip(w_dir, -0.1, 0.1)

    w_sub = mu_param * w_dir
    target[idx] = w_sub
    return target


## 5. Backtest

In [ ]:
def run_backtest(lambda_param: float, mu_param: float, capture_full: bool = True):
    """
    Q5 要求的 backtest：
      back_weight[t] = back_weight[t-1] * (1 + returns[t]) + trade[t]
      pnl_t = back_weight[t-1] · returns[t] - |trade_t|·tcost_t
    """
    lambda_param = float(np.clip(lambda_param, 0.0, 1.0))
    mu_param     = float(mu_param)

    positions = np.zeros((T, n))  # end-of-day positions in €
    trade     = np.zeros((T, n))
    pnl       = np.zeros(T)
    daily_pnl = np.zeros(T)
    booksize  = np.zeros(T)
    tradesize = np.zeros(T)
    long_side  = np.zeros(T)
    short_side = np.zeros(T)

    for t in range(1, T):
        date_t   = dates[t]
        date_t_1 = dates[t-1]

        # --- 这一步是 Q5 里强调的：旧仓位先被当天收益乘一遍 ---
        pretrade_positions = positions[t-1]

        # 先算毛 PnL：用昨天收盘仓位 * 今天收益
        gross_pnl_t = float(np.dot(positions[t-1], returns_np[t]))

        # 不在交易窗口：只滚动仓位和 PnL，不下单
        if (date_t < start_date) or (date_t > end_date):
            positions[t]  = pretrade_positions        # 没有 trade，因此收盘 = pretrade
            daily_pnl[t]  = gross_pnl_t
            pnl[t]        = pnl[t-1] + gross_pnl_t
            booksize[t]   = np.abs(positions[t]).sum()
            tradesize[t]  = 0.0
            long_side[t]  = positions[t][positions[t] > 0].sum()
            short_side[t] = -positions[t][positions[t] < 0].sum()
            continue

        # 在交易窗口：先算目标仓位
        Sigma_df, _ = get_cov_for_date(date_t_1)
        if Sigma_df is None:
            target = positions[t-1].copy()
        else:
            alpha_vec   = alphablend_np[t-1]
            active_mask = active_np[t-1]
            target = build_target(alpha_vec, Sigma_df, active_mask, mu_param)

        # 理论 delta
        desired_delta = target - positions[t-1]
        # 实际成交 = lambda * delta
        trade_t = lambda_param * desired_delta

        # 交易成本：按当天交易额乘 tcost
        cost_t = np.abs(trade_t) * tcost_np[t]
        cost_t_sum = float(cost_t.sum())

        # 净 PnL = 毛 PnL - 成本
        net_pnl_t = gross_pnl_t - cost_t_sum
        daily_pnl[t] = net_pnl_t
        pnl[t] = pnl[t-1] + net_pnl_t

        # Q5 公式：收盘仓位 = 经过收益后的旧仓位 + trade
        positions[t] = pretrade_positions + trade_t
        trade[t] = trade_t

        booksize[t]  = np.abs(positions[t]).sum()
        tradesize[t] = np.abs(trade_t).sum()
        long_side[t]  = positions[t][positions[t] > 0].sum()
        short_side[t] = -positions[t][positions[t] < 0].sum()

    # 只在交易区间内算平均
    window = slice(start_idx, end_idx + 1)
    trade_window = tradesize[window]
    if trade_window.size > 1:
        avg_trade_ex_first = trade_window[1:].mean()
    elif trade_window.size == 1:
        avg_trade_ex_first = trade_window[0]
    else:
        avg_trade_ex_first = 0.0

    avg_long  = long_side[window].mean() if trade_window.size > 0 else 0.0
    avg_short = short_side[window].mean() if trade_window.size > 0 else 0.0

    result = {
        "lambda": lambda_param,
        "mu": mu_param,
        "avg_trade_ex_first": avg_trade_ex_first,
        "avg_long": avg_long,
        "avg_short": avg_short,
        "daily_pnl": pd.Series(daily_pnl, index=dates, name="daily_pnl"),
        "pnl": pd.Series(pnl, index=dates, name="pnl"),
        "booksize": pd.Series(booksize, index=dates, name="booksize"),
        "tradesize": pd.Series(tradesize, index=dates, name="tradesize"),
    }

    # 这里就是 Q5 要的 back_weight：end-of-day positions in €
    if capture_full:
        result["trade"] = pd.DataFrame(trade, index=dates, columns=allstocks)
        result["back_weight"] = pd.DataFrame(positions, index=dates, columns=allstocks)

    return result


In [204]:
lambda_candidates = np.linspace(0.25, 0.65, 5)
mu_candidates = [10000000, 15000000, 20000000]

search_rows = []
best_combo = None
best_score = np.inf

for lam in lambda_candidates:
    for mu in mu_candidates:
        res = run_backtest(lam, mu, capture_full=False)
        avg_trade = res["avg_trade_ex_first"]
        avg_long  = res["avg_long"]
        avg_short = res["avg_short"]

        score = (
            ((avg_trade - 15000000) / 15000000) ** 2
            + ((avg_long  - 50000000) / 50000000) ** 2
            + ((avg_short - 50000000) / 50000000) ** 2
        )

        search_rows.append({
            "lambda": lam,
            "mu": mu,
            "avg_trade_ex_first": avg_trade,
            "avg_long": avg_long,
            "avg_short": avg_short,
            "score": score,
        })

        if score < best_score:
            best_score = score
            best_combo = (lam, mu)

search_df = pd.DataFrame(search_rows).sort_values("score").reset_index(drop=True)
search_df.head(10)


,lambda,mu,avg_trade_ex_first,avg_long,avg_short,score
0,0.65,20000000,1.387476e+07,1.818500e+07,1.814348e+07,0.816440
1,0.55,20000000,1.216690e+07,1.779137e+07,1.774967e+07,0.866665
2,0.45,20000000,1.046640e+07,1.728143e+07,1.723959e+07,0.948849
3,0.35,20000000,8.709180e+06,1.656859e+07,1.652683e+07,1.071131
4,0.65,15000000,1.040607e+07,1.363875e+07,1.360761e+07,1.152415
5,0.55,15000000,9.125176e+06,1.334353e+07,1.331225e+07,1.229269
6,0.25,20000000,6.806083e+06,1.548324e+07,1.544233e+07,1.252657
7,0.45,15000000,7.849801e+06,1.296107e+07,1.292969e+07,1.325660
8,0.35,15000000,6.531885e+06,1.242644e+07,1.239513e+07,1.449066
9,0.25,15000000,5.104562e+06,1.161243e+07,1.158175e+07,1.615025


In [205]:
print(f"Chosen lambda = {best_combo[0]:.3f}, mu = {best_combo[1]/1e6:.1f}M")

final_results = run_backtest(best_combo[0], best_combo[1], capture_full=True)

trade       = final_results["trade"]
back_weight = final_results["back_weight"]
pnl         = final_results["pnl"]
booksize    = final_results["booksize"]
tradesize   = final_results["tradesize"]
daily_pnl   = final_results["daily_pnl"]

lambda_value = final_results["lambda"]
mu_value     = final_results["mu"]

lambda_scalar = np.array([[lambda_value]])
mu_scalar     = np.array([[mu_value]])

print(f"平均日成交（剔除第一天）：{final_results['avg_trade_ex_first']/1e6:.2f}M")
print(f"平均多头 book：{final_results['avg_long']/1e6:.2f}M")
print(f"平均空头 book：{final_results['avg_short']/1e6:.2f}M")


Chosen lambda = 0.650, mu = 20.0M
平均日成交（剔除第一天）：13.87M
平均多头 book：18.18M
平均空头 book：18.14M


## 6. Evaluation

In [206]:
daily_pnl_nonzero = daily_pnl.iloc[1:]
pnl_std = daily_pnl_nonzero.std()

if pnl_std > 0:
    sharpe = float(np.sqrt(252) * daily_pnl_nonzero.mean() / pnl_std)
else:
    sharpe = 0.0

print("Annualized Sharpe =", sharpe)

cum_pnl = pnl
hwm = cum_pnl.cummax()
drawdown = hwm - cum_pnl

deepest_dd = float(drawdown.max())

drawdown_flags = drawdown > 1e-8
longest_dd = 0
current_run = 0
for flag in drawdown_flags:
    if flag:
        current_run += 1
        longest_dd = max(longest_dd, current_run)
    else:
        current_run = 0

print("Deepest drawdown =", deepest_dd)
print("Longest drawdown (days) =", longest_dd)

sharpe_val     = np.array([[sharpe]])
longest_dd_val = np.array([[longest_dd]])
deepest_dd_val = np.array([[deepest_dd]])


Annualized Sharpe = -4.053878937855116
Deepest drawdown = 20646393.204640187
Longest drawdown (days) = 1178


In [ ]:
import numpy as np
import pickle

# --- helper: Series -> (T×1) numpy array ---
def series_to_T1(s):
    return s.to_numpy(dtype=float).reshape(-1, 1)

# --- helper: scalar -> (1×1) numpy array ---
def scalar_to_1x1(x):
    return np.array([[float(x)]], dtype=float)

# shrink: 60×1
shrink_out = np.asarray(shrink, dtype=float).reshape(-1, 1)

# alphas: T×n
alpharev_out   = alpharev.to_numpy(dtype=float)
alpharec_out   = alpharec.to_numpy(dtype=float)
alphaval_out   = alphaval.to_numpy(dtype=float)
alphamom_out   = alphamom.to_numpy(dtype=float)
alphablend_out = alphablend.to_numpy(dtype=float)

# scalars: 1×1
lambda_out    = scalar_to_1x1(lambda_value)
mu_out        = scalar_to_1x1(mu_value)
t0_out        = scalar_to_1x1(t0)
sharpe_out    = scalar_to_1x1(sharpe)
longest_dd_out = scalar_to_1x1(longest_dd)
deepest_dd_out = scalar_to_1x1(deepest_dd)

# time series: T×1
pnl_out        = series_to_T1(pnl)
booksize_out   = series_to_T1(booksize)
tradesize_out  = series_to_T1(tradesize)

# trade / back_weight: T×n
trade_out       = trade.to_numpy(dtype=float)
back_weight_out = back_weight.to_numpy(dtype=float)

# --- 打包成 dict，key 名必须和要求一样 ---
results = {
    "shrink":      shrink_out,       # (60×1)
    "alpharev":    alpharev_out,     # (T×n)
    "alpharec":    alpharec_out,     # (T×n)
    "alphaval":    alphaval_out,     # (T×n)
    "alphamom":    alphamom_out,     # (T×n)
    "alphablend":  alphablend_out,   # (T×n)
    "lambda":      lambda_out,       # (1×1)
    "mu":          mu_out,           # (1×1)
    "t0":          t0_out,           # (1×1)
    "trade":       trade_out,        # (T×n)
    "back_weight": back_weight_out,  # (T×n)
    "pnl":         pnl_out,          # (T×1)
    "booksize":    booksize_out,     # (T×1)
    "tradesize":   tradesize_out,    # (T×1)
    "sharpe":      sharpe_out,       # (1×1)
    "longest_dd":  longest_dd_out,   # (1×1)
    "deepest_dd":  deepest_dd_out,   # (1×1)
}

# --- 保存为 pickle 文件（文件名你可以自己改，比如 ps_output.pkl）---
output_path = "ps3_output.pkl"
with open(output_path, "wb") as f:
    pickle.dump(results, f)

print("Saved problem set output to:", output_path)


Saved problem set output to: ps_output.pkl
